# KOCOH 제안 모델 - RoBERTa 기반 맥락 결합 분류 모델 (Colab 버전)

- 입력: `[CLS] context [SEP] comment [SEP]`
- 출력: 혐오(1) / 비혐오(0)

**실행 전 준비:**
1. 런타임 → 런타임 유형 변경 → **T4 GPU** 선택
2. 왼쪽 파일 아이콘 클릭 → `team_train.csv`, `team_valid.csv`, `team_test.csv` 업로드

## 1. 라이브러리 설치 및 기본 설정

In [1]:
!pip install transformers -q

In [2]:
import os
import random
import numpy as np
import pandas as pd
import torch

from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModelForSequenceClassification

SEED = 42
MAX_LENGTH = 256   # context + comment 합치므로 베이스라인(128)보다 늘림
BATCH_SIZE = 16    # Colab GPU 환경이므로 배치 크기 늘림
EPOCHS = 5         # 베이스라인과 동일 조건

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("사용 장치:", device)
print("MAX_LENGTH:", MAX_LENGTH)
print("EPOCHS:", EPOCHS)
print("BATCH_SIZE:", BATCH_SIZE)

사용 장치: cuda
MAX_LENGTH: 256
EPOCHS: 5
BATCH_SIZE: 16


## 2. 데이터 불러오기

왼쪽 파일 아이콘에서 `team_train.csv`, `team_valid.csv`, `team_test.csv` 업로드 후 실행하세요.

In [4]:
train_df = pd.read_csv("/content/team_train.csv")
valid_df = pd.read_csv("/content/team_valid.csv")
test_df  = pd.read_csv("/content/team_test.csv")

print("train:", train_df.shape)
print("valid:", valid_df.shape)
print("test :", test_df.shape)

display(train_df.head())

print("\n라벨 분포 (train):")
print(train_df["label"].value_counts())

train: (2395, 4)
valid: (299, 4)
test : (300, 4)


,context,comment,label,type
0,"고등학교 축제 나락퀴즈쇼 코너에서 ""가장 쓸모없다고 생각하는 운동을 고르시오 ①3·...",보기 숫자 위치 킹받네,0,3
1,50대 여성이 또래 남성에게 수면제를 먹여 성폭행을 시도했다.,증거 다 있는데 저렇게 수사가 늑장이면 경찰이 문제지,0,3
2,콴다 앱에서 고등학생의 질문에 대학생이 고등학생이 아직 배우지 않은 개념을 활용해 ...,대학보고 납득 ㅋㅋㅋ,1,1
3,"최근 벌어진 신종 로맨스스캠 사건의 피해자들은 대부분 40대 이상 독신 남성으로, ...",전주 ㅋㅋㅋㅋ,1,1
4,여자 형제로 추정되는 질문자가 엄마 아들의 피규어를 처분하는 방법을 물었다.,참 매정하네\n차갑다,0,3



라벨 분포 (train):
label
0    1784
1     611
Name: count, dtype: int64


## 3. Dataset 클래스 (context + comment 결합 입력)

In [5]:
class ContextCommentDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length=256):
        self.contexts  = dataframe["context"].astype(str).tolist()
        self.comments  = dataframe["comment"].astype(str).tolist()
        self.labels    = dataframe["label"].astype(int).tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        # tokenizer에 두 문장 넣으면 자동으로
        # [CLS] context [SEP] comment [SEP] 형태로 인코딩됨
        encoding = self.tokenizer(
            self.contexts[idx],
            self.comments[idx],
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )

        return {
            "input_ids":      encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels":         torch.tensor(self.labels[idx], dtype=torch.long)
        }

## 4. 평가 함수

In [6]:
def evaluate_model(model, dataloader):
    model.eval()

    all_preds  = []
    all_labels = []
    total_loss = 0

    with torch.no_grad():
        for batch in dataloader:
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels         = batch["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            total_loss += outputs.loss.item()
            preds = torch.argmax(outputs.logits, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(dataloader)
    acc = accuracy_score(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average="binary", zero_division=0
    )

    return {
        "loss":      avg_loss,
        "accuracy":  acc,
        "precision": precision,
        "recall":    recall,
        "f1":        f1,
        "y_true":    all_labels,
        "y_pred":    all_preds
    }

## 5. 제안 모델 학습 함수

In [7]:
def train_proposal_model(model_name, output_name):
    print("=" * 70)
    print("학습 모델:", model_name)
    print("입력 방식: context + comment 결합")
    print("=" * 70)

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
    model.to(device)

    train_dataset = ContextCommentDataset(train_df, tokenizer, MAX_LENGTH)
    valid_dataset = ContextCommentDataset(valid_df, tokenizer, MAX_LENGTH)
    test_dataset  = ContextCommentDataset(test_df,  tokenizer, MAX_LENGTH)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False)
    test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

    # 클래스 불균형 보정 (베이스라인과 동일 방식)
    label_counts = train_df["label"].value_counts().sort_index()
    total_count  = len(train_df)

    weight_0 = total_count / (2 * label_counts[0])
    weight_1 = total_count / (2 * label_counts[1])

    class_weights = torch.tensor([weight_0, weight_1], dtype=torch.float).to(device)
    print("Class weights:", class_weights)

    loss_fn   = torch.nn.CrossEntropyLoss(weight=class_weights)
    optimizer = AdamW(model.parameters(), lr=2e-5)

    for epoch in range(EPOCHS):
        model.train()
        total_train_loss = 0

        for step, batch in enumerate(train_loader, start=1):
            optimizer.zero_grad()

            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels         = batch["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            logits = outputs.logits
            loss   = loss_fn(logits, labels)

            total_train_loss += loss.item()
            loss.backward()
            optimizer.step()

            if step % 50 == 0:
                print(f"Epoch {epoch+1}/{EPOCHS} | Step {step}/{len(train_loader)} | Loss: {loss.item():.4f}")

        avg_train_loss = total_train_loss / len(train_loader)
        valid_result   = evaluate_model(model, valid_loader)

        print(f"\nEpoch {epoch+1}/{EPOCHS} 결과")
        print(f"Train Loss     : {avg_train_loss:.4f}")
        print(f"Valid Accuracy : {valid_result['accuracy']:.4f}")
        print(f"Valid Precision: {valid_result['precision']:.4f}")
        print(f"Valid Recall   : {valid_result['recall']:.4f}")
        print(f"Valid F1       : {valid_result['f1']:.4f}")
        print("-" * 70)

    test_result = evaluate_model(model, test_loader)

    print("\n===== TEST RESULT =====")
    print(f"Accuracy : {test_result['accuracy']:.4f}")
    print(f"Precision: {test_result['precision']:.4f}")
    print(f"Recall   : {test_result['recall']:.4f}")
    print(f"F1-score : {test_result['f1']:.4f}")

    print("\n===== Classification Report =====")
    print(classification_report(test_result["y_true"], test_result["y_pred"], digits=4))

    print("\n===== Confusion Matrix =====")
    print(confusion_matrix(test_result["y_true"], test_result["y_pred"]))

    return {
        "model":     output_name + "_context+comment",
        "accuracy":  test_result["accuracy"],
        "precision": test_result["precision"],
        "recall":    test_result["recall"],
        "f1":        test_result["f1"]
    }

## 6. 제안 모델: RoBERTa 기반 맥락 결합 분류 모델

In [8]:
roberta_proposal_result = train_proposal_model(
    model_name="klue/roberta-base",
    output_name="RoBERTa_context+comment"
)

roberta_proposal_result

학습 모델: klue/roberta-base
입력 방식: context + comment 결합


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/546 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/375 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/443M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: klue/roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Class weights: tensor([0.6712, 1.9599], device='cuda:0')
Epoch 1/5 | Step 50/150 | Loss: 0.5402
Epoch 1/5 | Step 100/150 | Loss: 0.3679
Epoch 1/5 | Step 150/150 | Loss: 0.3668

Epoch 1/5 결과
Train Loss     : 0.4800
Valid Accuracy : 0.8829
Valid Precision: 0.7531
Valid Recall   : 0.8026
Valid F1       : 0.7771
----------------------------------------------------------------------
Epoch 2/5 | Step 50/150 | Loss: 0.1125
Epoch 2/5 | Step 100/150 | Loss: 0.4381
Epoch 2/5 | Step 150/150 | Loss: 0.1898

Epoch 2/5 결과
Train Loss     : 0.2658
Valid Accuracy : 0.9030
Valid Precision: 0.8052
Valid Recall   : 0.8158
Valid F1       : 0.8105
----------------------------------------------------------------------
Epoch 3/5 | Step 50/150 | Loss: 0.1628
Epoch 3/5 | Step 100/150 | Loss: 0.0654
Epoch 3/5 | Step 150/150 | Loss: 0.1156

Epoch 3/5 결과
Train Loss     : 0.1851
Valid Accuracy : 0.9130
Valid Precision: 0.8378
Valid Recall   : 0.8158
Valid F1       : 0.8267
------------------------------------------

{'model': 'RoBERTa_context+comment_context+comment',
 'accuracy': 0.9266666666666666,
 'precision': 0.8160919540229885,
 'recall': 0.922077922077922,
 'f1': 0.8658536585365854}

## 7. 제안 모델 결과 저장

In [9]:
proposal_df = pd.DataFrame([roberta_proposal_result])

display(proposal_df)

proposal_df.to_csv("/content/proposal_result_roberta.csv", index=False, encoding="utf-8-sig")
print("결과 저장 완료: proposal_result_roberta.csv")

,model,accuracy,precision,recall,f1
0,RoBERTa_context+comment_context+comment,0.926667,0.816092,0.922078,0.865854


결과 저장 완료: proposal_result_roberta.csv


## 8. 전체 모델 성능 비교표

In [10]:
# 기존 KOCOH 논문 LLM 결과 (논문 인용값)
llm_result = {
    "model":     "KOCOH 논문 LLM 평균 (GPT-4o 등 6개)",
    "accuracy":  "-",
    "precision": "-",
    "recall":    "-",
    "f1":        0.6073
}

# 베이스라인 실제 결과
bert_baseline_result = {
    "model":     "BERT_comment_only_class_weight",
    "accuracy":  0.5967,
    "precision": 0.3472,
    "recall":    0.6494,
    "f1":        0.4525
}

roberta_baseline_result = {
    "model":     "RoBERTa_comment_only_class_weight",
    "accuracy":  0.6367,
    "precision": 0.3710,
    "recall":    0.5974,
    "f1":        0.4578
}

# 전체 비교표
compare_df = pd.DataFrame([
    llm_result,
    bert_baseline_result,
    roberta_baseline_result,
    roberta_proposal_result
])

print("=" * 70)
print("전체 모델 성능 비교표")
print("=" * 70)
display(compare_df)

compare_df.to_csv("/content/final_compare_result.csv", index=False, encoding="utf-8-sig")
print("\n비교표 저장 완료: final_compare_result.csv")

전체 모델 성능 비교표


,model,accuracy,precision,recall,f1
0,KOCOH 논문 LLM 평균 (GPT-4o 등 6개),-,-,-,0.607300
1,BERT_comment_only_class_weight,0.5967,0.3472,0.6494,0.452500
2,RoBERTa_comment_only_class_weight,0.6367,0.371,0.5974,0.457800
3,RoBERTa_context+comment_context+comment,0.926667,0.816092,0.922078,0.865854



비교표 저장 완료: final_compare_result.csv


## 9. 결과 파일 다운로드

In [11]:
from google.colab import files
files.download("/content/proposal_result_roberta.csv")
files.download("/content/final_compare_result.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>